# Z-Score Normalization

구글 드라이브 마운트 및 경로 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'
ratings = pd.read_csv(f'{DATA_PATH}/ratings.csv')
movies  = pd.read_csv(f'{DATA_PATH}/movies.csv')

SEED_LIST = [10, 21, 35, 42, 57, 60, 73, 88, 95, 101]
PERSONA_HEAVY_USER = 414
PERSONA_GENRE_SPECIALIST = 85

## 1. 전처리 함수 (Z-Score + 최소 평점 필터링)

In [3]:
def run_preprocessing(train_df, test_df):
    train = train_df.copy()
    test  = test_df.copy()

    # 1. 최소 5회 이상 평점 준 사용자만 사용 (페르소나는 강제 포함)
    user_counts = train['userId'].value_counts()
    valid_users = set(user_counts[user_counts >= 5].index)
    valid_users = valid_users.union({PERSONA_HEAVY_USER, PERSONA_GENRE_SPECIALIST})

    train = train[train['userId'].isin(valid_users)]
    test  = test[test['userId'].isin(valid_users)]

    # 2. Train에서만 사용자별 mean / std 계산
    user_stats = train.groupby('userId')['rating'].agg(['mean', 'std']).fillna(1.0)
    user_stats['std'] = user_stats['std'].replace(0, 1.0)   # std=0 방지

    # 3. Z-Score 변환 (Train)
    train = train.merge(user_stats, on='userId', suffixes=('', '_user'))
    train['z_rating'] = (train['rating'] - train['mean']) / train['std']

    # 4. Test도 동일한 통계량으로 변환 (Train에 없는 사용자는 제거 → cold-start 방지)
    test = test.merge(user_stats, on='userId', how='left')
    test = test.dropna(subset=['mean', 'std'])   # Train에 없는 사용자 제거
    test['z_rating'] = (test['rating'] - test['mean']) / test['std']

    return train, test, user_stats   # user_stats도 반환 (역변환에 필요)

## 2. 평가 함수

In [4]:
def rmse(y_true, y_pred): return np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2))
def mae(y_true, y_pred):  return np.mean(np.abs(np.array(y_true)-np.array(y_pred)))

def time_predictions(fn, pairs):
    s = time.perf_counter()
    preds = [fn(u,i) for u,i in pairs]
    e = time.perf_counter() - s
    return preds, e, (e/len(pairs))*1000

## 3. Multi-seed 실험 루프

### 3.1 baseline

In [5]:

results = []
best_reco = None

for idx, SEED in enumerate(SEED_LIST):
    print(f"\nRunning Seed {SEED} ({idx+1}/{len(SEED_LIST)})")

    train_raw, test_raw = train_test_split(ratings, test_size=0.2, random_state=SEED)
    train, test, user_stats = run_preprocessing(train_raw, test_raw)

    # Z-Score된 rating으로 행렬 구성
    ui = train.pivot(index='userId', columns='movieId', values='z_rating').fillna(0)
    user_ids  = ui.index.tolist()
    movie_ids = ui.columns.tolist()
    ui_mat = ui.values

    user_to_idx = {u:i for i,u in enumerate(user_ids)}
    movie_to_idx = {m:i for i,m in enumerate(movie_ids)}

    # Item-Item 유사도 (dot product on Z-score centered matrix)
    def get_neighbors(midx, K=40):
        sims = np.dot(ui_mat.T, ui_mat[:, midx])
        sims[midx] = -np.inf
        topk = np.argsort(-sims)[:K]
        return [(int(j), float(sims[j])) for j in topk if sims[j] > 0]

    global_z_mean = train['z_rating'].mean()

    def predict_z(user_id, movie_id):
        if user_id not in user_to_idx or movie_id not in movie_to_idx:
            return global_z_mean
        uidx = user_to_idx[user_id]
        midx = movie_to_idx[movie_id]
        neighbors = get_neighbors(midx)
        num, den = 0.0, 0.0
        for j_idx, sim in neighbors:
            if ui.iat[uidx, j_idx] == 0: continue
            r_z = ui.iat[uidx, j_idx]
            num += sim * r_z
            den += abs(sim)
        return 0.0 if den == 0 else num / den

    # Test 샘플링 (최대 3000개)
    rng = np.random.default_rng(SEED)
    test_sample = test[['userId', 'movieId', 'rating', 'mean', 'std']].sample(n=min(3000, len(test)), random_state=SEED)

    pairs = list(zip(test_sample['userId'], test_sample['movieId']))
    z_preds, elapsed, avg_ms = time_predictions(predict_z, pairs)

    # 역변환해서 원래 스케일로 되돌리기
    original_preds = []
    for pred_z, row in zip(z_preds, test_sample.itertuples()):
        mean_u = row.mean
        std_u  = row.std
        original_preds.append(pred_z * std_u + mean_u)

    res = {
        'seed': SEED,
        'rmse': rmse(test_sample['rating'], original_preds),
        'mae' : mae(test_sample['rating'], original_preds),
        'avg_ms': avg_ms
    }
    results.append(res)

    # 첫 번째 시드에서만 추천 결과 저장
    if idx == 0:
        def recommend(user_id, topk=3):
            if user_id not in user_to_idx: return None
            watched = set(train[train['userId']==user_id]['movieId'])
            candidates = [m for m in movie_ids if m not in watched]
            scores = []
            for m in candidates:
                z = predict_z(user_id, m)
                mean_u = user_stats.loc[user_id, 'mean']
                std_u  = user_stats.loc[user_id, 'std']
                scores.append((m, z * std_u + mean_u))
            scores.sort(key=lambda x: -x[1])
            top = scores[:topk]
            return [{
                'movieId': m,
                'title': movies.loc[movies['movieId']==m, 'title'].iloc[0],
                'predicted_rating': round(score, 3)
            } for m, score in top]

        best_reco = {
            'heavy': recommend(PERSONA_HEAVY_USER),
            'specialist': recommend(PERSONA_GENRE_SPECIALIST)
        }

print("\n실험 완료!")


Running Seed 10 (1/10)

Running Seed 21 (2/10)

Running Seed 35 (3/10)

Running Seed 42 (4/10)

Running Seed 57 (5/10)

Running Seed 60 (6/10)

Running Seed 73 (7/10)

Running Seed 88 (8/10)

Running Seed 95 (9/10)

Running Seed 101 (10/10)

실험 완료!


### 3.2 Z-score기반 추천 모델 가중치 적용 실험 (4가지)  
매니아 점수에 +,- 가중치  
헤비유저 점수에 +,- 가중치

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 실험 설정 정의 (케이스 이름, 대상 유저 ID, 가중치)
experiment_configs = [
    ("매니아_강조", PERSONA_GENRE_SPECIALIST, 1.5),  # 85번 유저 점수 1.5배 (영향력 확대)
    ("매니아_약화", PERSONA_GENRE_SPECIALIST, 0.5),  # 85번 유저 점수 0.5배 (영향력 축소)
    ("헤비유저_강조", PERSONA_HEAVY_USER, 1.5),      # 414번 유저 점수 1.5배
    ("헤비유저_약화", PERSONA_HEAVY_USER, 0.5)       # 414번 유저 점수 0.5배
]

# 결과 저장소
final_summary = []
persona_reco_results = {} # 각 케이스별 페르소나 추천 결과 저장용


# 2. 실험 루프
for exp_name, target_user, weight in experiment_configs:
    print(f"\n{'='*60}")
    print(f"▶ 실험 시작: {exp_name} (User: {target_user}, Weight: {weight})")
    print(f"{'='*60}")

    temp_rmse, temp_mae, temp_time = [], [], []

    # 추천 결과 비교를 위해 첫 번째 시드(Seed 10)에서만 추천 리스트 추출
    reco_extracted = False

    for idx, SEED in enumerate(SEED_LIST):
        # A. 전처리
        train_raw, test_raw = train_test_split(ratings, test_size=0.2, random_state=SEED)
        train, test, user_stats = run_preprocessing(train_raw, test_raw)


        # B. 가중치 적용 (단일 유저 대상)
        train_weighted = train.copy()

        # 지정된 1명의 점수에만 가중치 적용
        mask = train_weighted['userId'] == target_user
        train_weighted.loc[mask, 'z_rating'] *= weight

        # 유사도 계산용 행렬 (가중치 O)
        ui_weighted = train_weighted.pivot(index='userId', columns='movieId', values='z_rating').fillna(0)
        ui_mat_weighted = ui_weighted.values

        # 예측값 참조용 행렬 (원본)
        ui = train.pivot(index='userId', columns='movieId', values='z_rating').fillna(0)

        user_ids  = ui.index.tolist()
        movie_ids = ui.columns.tolist()
        user_to_idx = {u:i for i,u in enumerate(user_ids)}
        movie_to_idx = {m:i for i,m in enumerate(movie_ids)}

        # 모델 구성
        ui_mat = ui_mat_weighted # 가중치 적용된 행렬 사용

        def get_neighbors(midx, K=40):
            sims = np.dot(ui_mat.T, ui_mat[:, midx])
            sims[midx] = -np.inf
            topk = np.argsort(-sims)[:K]
            return [(int(j), float(sims[j])) for j in topk if sims[j] > 0]

        global_z_mean = train['z_rating'].mean()

        def predict_z(user_id, movie_id):
            if user_id not in user_to_idx or movie_id not in movie_to_idx:
                return global_z_mean
            uidx = user_to_idx[user_id]
            midx = movie_to_idx[movie_id]
            neighbors = get_neighbors(midx)

            num, den = 0.0, 0.0
            for j_idx, sim in neighbors:
                if ui.iat[uidx, j_idx] == 0: continue
                r_z = ui.iat[uidx, j_idx] # 원본 점수 참조
                num += sim * r_z
                den += abs(sim)
            return 0.0 if den == 0 else num / den


        # C. 추천 리스트 뽑기 (첫 번째 시드에서만 수행)
        if idx == 0 and not reco_extracted:
            def recommend_top3(u_id):
                if u_id not in user_to_idx: return []
                watched = set(train[train['userId']==u_id]['movieId'])
                candidates = [m for m in movie_ids if m not in watched]
                scores = []
                for m in candidates:
                    z = predict_z(u_id, m)
                    mean_u = user_stats.loc[u_id, 'mean']
                    std_u  = user_stats.loc[u_id, 'std']
                    scores.append((m, z * std_u + mean_u))
                scores.sort(key=lambda x: -x[1])
                return scores[:3]

            # 현재 실험 대상 유저에게 추천되는 Top3 영화 저장
            top3 = recommend_top3(target_user)
            top3_titles = [movies.loc[movies['movieId']==m, 'title'].iloc[0] for m, s in top3]
            persona_reco_results[exp_name] = top3_titles
            reco_extracted = True

        # D. 평가 (RMSE, MAE)
        test_sample = test[['userId', 'movieId', 'rating', 'mean', 'std']].sample(n=min(3000, len(test)), random_state=SEED)
        pairs = list(zip(test_sample['userId'], test_sample['movieId']))
        z_preds, elapsed, avg_ms = time_predictions(predict_z, pairs)

        original_preds = []
        for pred_z, row in zip(z_preds, test_sample.itertuples()):
            original_preds.append(pred_z * row.std + row.mean)

        temp_rmse.append(rmse(test_sample['rating'], original_preds))
        temp_mae.append(mae(test_sample['rating'], original_preds))
        temp_time.append(avg_ms)

        print(f"   [Seed {SEED}] RMSE: {temp_rmse[-1]:.4f}", end='\r')

    # 결과 요약 저장
    final_summary.append({
        'Case': exp_name,
        'RMSE': np.mean(temp_rmse),
        'MAE': np.mean(temp_mae),
        'Time(ms)': np.mean(temp_time)
    })
    print(f"\n   -> {exp_name} 완료 (RMSE: {np.mean(temp_rmse):.4f})")


# 3. 통합 결과 비교 및 추천 리스트 변화 확인
if 'results' in globals() and len(results) > 0:
    df_base = pd.DataFrame(results)
    base_rmse = df_base['rmse'].mean()
    base_mae = df_base['mae'].mean()
    base_time = df_base['avg_ms'].mean()
else:
    base_rmse, base_mae, base_time = 0.0, 0.0, 0.0

summary_data = []
summary_data.append({
    'Case': 'Baseline (기준)',
    'RMSE': base_rmse,
    'MAE': base_mae,
    'Time(ms)': base_time,
    'vs Base': 0.0
})

for item in final_summary:
    summary_data.append({
        'Case': item['Case'],
        'RMSE': item['RMSE'],
        'MAE': item['MAE'],
        'Time(ms)': item['Time(ms)'],
        'vs Base': item['RMSE'] - base_rmse
    })

df_summary = pd.DataFrame(summary_data)

print("\n" + "="*60)
print("[ 실험 결과 1: 전체 성능 (RMSE) ]")
print("="*60)
print(df_summary[['Case', 'RMSE', 'vs Base', 'MAE']].round(4))

print("\n" + "="*60)
print("[ 실험 결과 2: 추천 리스트 변화 (Top 3) ]")
print(": 가중치에 따라 추천되는 영화가 바뀌는지 확인")
print("="*60)

# Baseline 추천 결과
if 'best_reco' in globals() and best_reco is not None:
    print(f"▶ Baseline (원본)")
    print(f"   - 헤비유저(414): {[x['title'] for x in best_reco['heavy']]}")
    print(f"   - 전문가(85):   {[x['title'] for x in best_reco['specialist']]}")
    print("-" * 40)

for exp_name, top3_list in persona_reco_results.items():
    print(f"▶ {exp_name}")
    print(f"   - 추천 결과: {top3_list}")


▶ 실험 시작: 매니아_강조 (User: 85, Weight: 1.5)
   [Seed 101] RMSE: 0.9228
   -> 매니아_강조 완료 (RMSE: 0.9205)

▶ 실험 시작: 매니아_약화 (User: 85, Weight: 0.5)
   [Seed 101] RMSE: 0.9227
   -> 매니아_약화 완료 (RMSE: 0.9204)

▶ 실험 시작: 헤비유저_강조 (User: 414, Weight: 1.5)
   [Seed 101] RMSE: 0.9221
   -> 헤비유저_강조 완료 (RMSE: 0.9232)

▶ 실험 시작: 헤비유저_약화 (User: 414, Weight: 0.5)
   [Seed 101] RMSE: 0.9264
   -> 헤비유저_약화 완료 (RMSE: 0.9210)

[ 실험 결과 1: 전체 성능 (RMSE) ]
            Case    RMSE  vs Base     MAE
0  Baseline (기준)  0.9205   0.0000  0.6934
1         매니아_강조  0.9205  -0.0000  0.6934
2         매니아_약화  0.9204  -0.0002  0.6933
3        헤비유저_강조  0.9232   0.0026  0.6957
4        헤비유저_약화  0.9210   0.0004  0.6934

[ 실험 결과 2: 추천 리스트 변화 (Top 3) ]
: 가중치에 따라 추천되는 영화가 바뀌는지 확인
▶ Baseline (원본)
   - 헤비유저(414): ['Harmonists, The (1997)', 'Happy Go Lovely (1951)', 'Hawks and Sparrows (Uccellacci e Uccellini) (1966)']
   - 전문가(85):   ['Toy Story (1995)', 'Heat (1995)', 'American President, The (1995)']
-----------------------------------

## 4. 통합 결과 요약

In [7]:
import pandas as pd

# 1. Baseline 결과 정리
df_base = pd.DataFrame(results)
base_rmse = df_base['rmse'].mean()
base_mae = df_base['mae'].mean()
base_time = df_base['avg_ms'].mean()

print("="*60)
print(f"BASELINE 성능 (평균)")
print(f"RMSE : {base_rmse:.4f}  |  MAE : {base_mae:.4f}  |  속도 : {base_time:.2f} ms")
print("="*60)

# 2. 비교 테이블 생성
summary_data = []

# (1) Baseline 행 추가
summary_data.append({
    'Case': 'Baseline (기준)',
    'RMSE': base_rmse,
    'MAE': base_mae,
    'Time(ms)': base_time,
    'vs Base (RMSE)': 0.0  # 기준점이므로 0
})

# (2) 가중치 실험 행 추가
for item in final_summary:
    summary_data.append({
        'Case': item['Case'],
        'RMSE': item['RMSE'],
        'MAE': item['MAE'],
        'Time(ms)': item['Time(ms)'],
        'vs Base (RMSE)': item['RMSE'] - base_rmse # 차이 계산
    })

# 3. DataFrame 출력
df_summary = pd.DataFrame(summary_data)
# 보기 좋게 컬럼 순서 정렬 및 반올림
cols = ['Case', 'RMSE', 'vs Base (RMSE)', 'MAE', 'Time(ms)']
print("\n[ 실험 결과 비교 요약 ]")
print(df_summary[cols].round(4).to_string(index=False))

print("\n" + "-"*60)
print("※ 'vs Base (RMSE)'가 마이너스(-)일수록 성능이 개선된 것")
print("-"*60)

BASELINE 성능 (평균)
RMSE : 0.9205  |  MAE : 0.6934  |  속도 : 4.58 ms

[ 실험 결과 비교 요약 ]
         Case   RMSE  vs Base (RMSE)    MAE  Time(ms)
Baseline (기준) 0.9205          0.0000 0.6934    4.5828
       매니아_강조 0.9205         -0.0000 0.6934    4.4995
       매니아_약화 0.9204         -0.0002 0.6933    4.3883
      헤비유저_강조 0.9232          0.0026 0.6957    4.4508
      헤비유저_약화 0.9210          0.0004 0.6934    4.4304

------------------------------------------------------------
※ 'vs Base (RMSE)'가 마이너스(-)일수록 성능이 개선된 것
------------------------------------------------------------


- 단일 유저(1명)에 대한 가중치 조절이 전체 모델의 성능(Global RMSE)에는 거의 영향을 주지 못함  
- 헤비 유저를 강조했을 때 성능이 가장 많이 하락  
1. 매니아(85번) 1명의 취향을 강화해도 전체 시스템에는 영향 X  
2. 매니아 악화 -> 굳이 따지자면 해당 유저의 독특한 취향을 배제하니 아주 조금 일반화 성능이 좋아진 것으로 추정됨  
3. 헤비유저(414번)의 데이터를 신뢰(강조)했더니 오히려 전체 예측력이 떨어짐  
4. 헤비유저의 영향력을 줄여도 성능이 소폭 하락함 (헤비유저 데이터가 완전히 쓸모없지는 않다는 뜻)  

-> 헤비 유저 취향을 많이 반영할수록 영화들 간의 뚜렷한 '유사성 패턴'을 흐리는 noise 역할을 했을 가능성이 큼  
-> 매니아 케이스의 변화가 0.0000인 것은, 해당 유저가 평가한 영화가 전체 데이터셋에서 차지하는 비중이 매우 작아 전체 RMSE 평균에 영향을 주지 못했기 때문  

성능 향상 방안: '단일 유저'가 아니라, '신뢰할 수 있는 유저 그룹' 전체에 가중치를 주는 방식 도입  
(ex. 평점 변동성이 커서 호불호가 확실한 상위 10% 유저 그룹 가중치 상향 등)